# Heatmap performa persona agama — ETHICS, 06–13

Acuan gender: judul 30/26 pt, angka sel 26 pt, label 22 pt, PNG. Lebar 30 inci menampung 17 skenario; jangan memaksakan kanvas gender yang lebih sempit. Nilai/arah, p-value dan keluarga Holm tetap. Semua perubahan berada dalam notebook, dengan staging dan arsip, tanpa menghapus permanen SVG lama. NA bukan nol.

In [1]:
BENCHMARK="ETHICS"
FOLDER="ethics accuracy analysis"
NOTEBOOK_NAME="ETHICS_Religion_Accuracy_Heatmap_06_13.ipynb"
from pathlib import Path
from datetime import datetime
import ast, hashlib, json, re, shutil, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.text import Text
from matplotlib.collections import PathCollection
from PIL import Image as PILImage
from IPython.display import Image, Markdown, display
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p/'AGENTS.md').is_file())
sys.path.insert(0,str(ROOT))
sys.path.insert(0,str(ROOT/'src/pipelines/analysis'))
ACTIVE = ROOT/'notebooks/religion'/FOLDER
TABLE_DIR = (ROOT/'result/benchmarks/gsm8k/reports/persona_publication_analysis_v5_paired_significance_six_models_bar3_compact_smalllabels_heatmap3_sigbold/tables') if BENCHMARK=='GSM8K' else ACTIVE/'tables'
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN = ROOT/'result/analyses/accuracy/runs'/('religion_'+BENCHMARK.lower()+'_presentation_'+RUN_ID)
FIGURE_DIR = RUN/'figures'
FIGURE_DIR.mkdir(parents=True,exist_ok=False)
def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
TABLE_HASHES = {str(p.resolve()):file_hash(p) for p in TABLE_DIR.glob('*.csv')}
LEGACY_STEMS = {'00_ethics_religion_combined_absolute_accuracy','00_ethics_religion_combined_inference_effect_holm','00_ethics_religion_combined_exact_mcnemar_four_panels','00_ethics_religion_combined_holm_pvalue_four_panels'}
MODEL_SOURCE = ['Qwen3 Next 80B-A3B','Llama 3.1 8B','Ministral 8B','DeepSeek R1-Distill 32B','GPT-OSS 20B','Llama 3.3 70B']
IDENTITIES = ['atheist','christian','islam','jewish']
IL = dict(zip(IDENTITIES,['Ateis','Kristen','Islam','Yahudi']))
PERSONAS = [i+'_'+c for i in IDENTITIES for c in ['explicit','implicit_1','implicit_2','implicit_3']]+['neutral']
PERSONA_LABELS = [IL[i]+' '+c for i in IDENTITIES for c in ['E','I1','I2','I3']]+['Neutral']
CATEGORIES = [IL[i]+' '+c for i in IDENTITIES for c in ['Explicit','Implicit Avg']]+['Neutral']
COLORS = ['#0047AB','#0086A8','#B80F2E','#E45700','#007A4D','#66A61E','#5B21B6','#A0006D','#30343B']
BACKGROUND = '#FCFBF8'
FONT_SIZES = dict(title=30,panel=26,axis=22,colorbar=25,cell=26)
RENDER_AUDIT = []
def display_label(value):
    s=str(value).replace('80B-A3B','80B')
    for old,new in [('DeepSeek R1-Distill','DeepSeek R1'),('Qwen3 Next 80B-A3B','Qwen3 Next 80B'),('Atheist','Ateis'),('Christian','Kristen'),('Jewish','Yahudi'),('Religion Persona & Baseline','Persona agama dan skenario netral'),('Religion','Agama'),('Global Neutral Avg','Rerata N enam model'),('Neutral Avg','Rerata N'),('Implicit Avg','Rerata I1–I3'),('Explicit','E'),('Implicit','I'),('Neutral','N'),('Accuracy','Akurasi'),('Difference on accuracy scale','Delta akurasi (skala 0–1)')]:
        s=s.replace(old,new)
    return s
def plot_values(fig):
    values=[]
    for ax in fig.axes:
        values += [np.array(line.get_xydata(),copy=True) for line in ax.lines]
        values += [np.array([p.get_x(),p.get_y(),p.get_width(),p.get_height()]) for p in ax.patches if hasattr(p,'get_height')]
        values += [np.array(c.get_array(),copy=True) for c in ax.collections if c.get_array() is not None]
    return values
def format_figure(fig):
    before=plot_values(fig)
    for ax in fig.axes:
        xlim,ylim=ax.get_xlim(),ax.get_ylim()
        for name in ['x','y']:
            texts=getattr(ax,'get_'+name+'ticklabels')()
            if any(display_label(t.get_text())!=t.get_text() for t in texts):
                getattr(ax,'set_'+name+'ticks')(getattr(ax,'get_'+name+'ticks')(),[display_label(t.get_text()) for t in texts])
        ax.set_xlim(xlim);ax.set_ylim(ylim)
        ax.title.set_fontweight('bold')
        for p in ax.patches:
            if hasattr(p,'get_height'): p.set_alpha(1)
        for c in ax.collections:
            if isinstance(c,PathCollection): c.set_alpha(1)
    for ax in fig.axes:
        for collection in ax.collections:
            a=collection.get_array()
            if a is not None and np.ndim(a)==2:
                for y,x in np.argwhere(np.ma.getmaskarray(a)):
                    ax.text(x+.5,y+.5,'NA',ha='center',va='center',fontsize=12,color='#17212B')
    for t in fig.findobj(Text): t.set_text(display_label(t.get_text()))
    if fig._suptitle: fig._suptitle.set_fontweight('bold')
    after=plot_values(fig)
    assert len(before)==len(after)
    assert all(np.array_equal(a,b,equal_nan=True) for a,b in zip(before,after))
    return len(before)
def save_plot(fig,stem,detail=''):
    arrays=format_figure(fig)
    path=FIGURE_DIR/(stem+'.png')
    assert not path.exists(),path
    fig.savefig(path,dpi=220,facecolor=BACKGROUND,bbox_inches='tight',pad_inches=.25)
    plt.close(fig)
    RENDER_AUDIT.append(dict(file=path.name,numeric_arrays_unchanged=arrays,detail=detail))
    print('PNG:',path.name,flush=True)
    return path
def show_figure(name,width=1300):
    if not name.endswith('.png'): name+='.png'
    display(Image(filename=str(ACTIVE/'figures'/name),width=width))
def publish():
    assert all(file_hash(p)==h for p,h in TABLE_HASHES.items())
    # Preserve PNG types not regenerated; invalid pooled accuracy remains in the archive.
    for p in (ACTIVE/'figures').glob('*.png'):
        if p.stem not in LEGACY_STEMS and not (FIGURE_DIR/p.name).exists():
            shutil.copy2(p,FIGURE_DIR/p.name)
    assert not list(FIGURE_DIR.glob('*.svg'))
    for p in FIGURE_DIR.glob('*.png'):
        with PILImage.open(p) as im: im.verify()
    archive=ACTIVE/'_arsip'/('accuracy_before_'+RUN_ID)
    archive.mkdir(parents=True,exist_ok=False)
    for target in [ACTIVE/'figures',archive]:
        assert target.resolve().is_relative_to((ROOT/'notebooks/religion').resolve())
    old={str(p.relative_to(ACTIVE/'figures')):file_hash(p) for p in (ACTIVE/'figures').rglob('*') if p.is_file()}
    shutil.move(str(ACTIVE/'figures'),str(archive/'figures'))
    assert old=={str(p.relative_to(archive/'figures')):file_hash(p) for p in (archive/'figures').rglob('*') if p.is_file()}
    for name in ['README.md','LATEST_RUN.json','output_index.csv','active_figures.csv','presentation_manifest.json','bundle_manifest.json']:
        if (ACTIVE/name).exists(): shutil.copy2(ACTIVE/name,archive/name)
    shutil.copy2(ACTIVE/NOTEBOOK_NAME,archive/NOTEBOOK_NAME)
    shutil.copytree(FIGURE_DIR,ACTIVE/'figures')
    index=pd.DataFrame([dict(kind='figure',file=p.name,relative_path='figures/'+p.name,path=str(ACTIVE/'figures'/p.name),sha256=file_hash(p),status='regenerated' if p.name in {r['file'] for r in RENDER_AUDIT} else 'preserved_supporting_png') for p in sorted(FIGURE_DIR.glob('*.png'))])
    index.to_csv(RUN/'figure_index.csv',index=False)
    index.to_csv(ACTIVE/'output_index.csv',index=False)
    index.rename(columns={'relative_path':'figure'}).to_csv(ACTIVE/'active_figures.csv',index=False)
    manifest=dict(benchmark=BENCHMARK,run=str(RUN),archive=str(archive),source_table_hashes=TABLE_HASHES,rendered=RENDER_AUDIT,png=len(index),svg=0,new_tests=False,rescoring=False,api_calls=0,latex_changed=False,legacy_pooled_accuracy_excluded=sorted(LEGACY_STEMS) if BENCHMARK=='ETHICS' else [])
    (RUN/'presentation_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    (ACTIVE/'presentation_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    (ACTIVE/'LATEST_RUN.json').write_text(json.dumps({'run':str(RUN),'archive':str(archive)},indent=2),encoding='utf-8')
    (archive/'preserved_hashes.json').write_text(json.dumps(old,indent=2),encoding='utf-8')
    assert all(file_hash(ACTIVE/'figures'/r['file'])==r['sha256'] for r in index.to_dict('records'))
    print('PUBLISH_OK',BENCHMARK,len(index),'PNG; sumber statistik tidak berubah.',flush=True)
    return manifest
sns.set_theme(style='whitegrid',font='DejaVu Sans')
plt.rcParams.update({'figure.facecolor':BACKGROUND,'axes.facecolor':BACKGROUND,'axes.titleweight':'bold','font.size':14,'axes.titlesize':23,'axes.labelsize':18,'xtick.labelsize':14,'ytick.labelsize':14,'legend.fontsize':13})
def render_parity_heatmaps():
    summary=pd.read_csv(TABLE_DIR/'ethics_condition_summary_source_derived.csv')
    paired=pd.read_csv(TABLE_DIR/'ethics_religion_parity_paired_effects_vs_neutral.csv')
    assert len(summary)==408 and int(summary.total.sum())==154428
    assert len(paired)==960 and paired.family_size.eq(16).all()
    assert np.allclose(paired.risk_difference_pp,paired.rate_a_pct-paired.rate_b_pct,equal_nan=True)
    assert set(paired.query("outcome in ['conditional_accuracy','effective_success']").subset)=={'commonsense','justice'}
    specs=[
      ('accuracy_committed',['commonsense','justice'],'RdYlGn','Akurasi committed per subset','06_ethics_religion_accuracy_committed_valid_subsets_heatmap','Akurasi committed (%)'),
      ('effective_success_rate_nonmissing',['commonsense','justice'],'RdYlGn','Keberhasilan pada respons non-missing','07_ethics_religion_effective_success_valid_subsets_heatmap','Benar / non-missing (%)'),
      ('pct_answer_wrong',['deontology','virtue'],'RdYlBu_r','Distribusi jawaban wrong','08_ethics_religion_stance_wrong_construct_invalid_heatmap','Wrong / committed (%)')]
    for metric,subsets,cmap,title,stem,unit in specs:
        fig,axes=plt.subplots(len(subsets),1,figsize=(30,8.7*len(subsets)),squeeze=False,layout='constrained')
        for ax,subset in zip(axes.flat,subsets):
            matrix=summary[summary.subset.eq(subset)].pivot(index='model_label',columns='persona_label',values=metric).reindex(index=MODEL_SOURCE,columns=PERSONA_LABELS)*100
            sns.heatmap(matrix,annot=True,fmt='.1f',cmap=cmap,vmin=0,vmax=100,annot_kws={'fontsize':26,'fontweight':'normal'},cbar_kws={'label':unit},ax=ax)
            ax.set_title(subset.title(),fontsize=26,fontweight='bold')
            ax.set_xlabel('');ax.set_ylabel('');ax.tick_params(axis='x',rotation=42,labelsize=22);ax.tick_params(axis='y',rotation=0,labelsize=22)
            ax.collections[0].colorbar.ax.tick_params(labelsize=22)
            ax.collections[0].colorbar.ax.yaxis.label.set_size(25)
        fig.suptitle('ETHICS · Persona agama: '+title,fontsize=30,fontweight='bold')
        fig.supxlabel('N: netral · E: eksplisit · I1–I3: implisit · coverage dan denominator: tabel notebook / grafik 03',fontsize=16)
        save_plot(fig,stem,'Deskriptif; denominator eksplisit')
    for outcome,title,number in [('coverage','Delta commit rate',10),('conditional_accuracy','Delta akurasi pada pasangan committed',11),('effective_success','Delta keberhasilan pada pasangan non-missing',12),('stance_wrong','Delta proporsi jawaban wrong',13)]:
        d=paired[paired.outcome.eq(outcome)]
        subsets=[s for s in ['commonsense','deontology','justice','virtue'] if s in set(d.subset)]
        limit=max(1,float(d.risk_difference_pp.abs().max()))
        fig,axes=plt.subplots(len(subsets),1,figsize=(30,8.7*len(subsets)),squeeze=False,layout='constrained')
        for ax,subset in zip(axes.flat,subsets):
            part=d[d.subset.eq(subset)]
            values=part.pivot(index='model_label',columns='persona_label',values='risk_difference_pp').reindex(index=MODEL_SOURCE,columns=PERSONA_LABELS[:-1])
            sig=part.pivot(index='model_label',columns='persona_label',values='significant_holm_0_05').reindex_like(values)
            ann=np.array([['NA' if pd.isna(v) else f'{v:+.1f}'+('*' if bool(s) else '') for v,s in zip(vr,sr)] for vr,sr in zip(values.to_numpy(),sig.to_numpy())])
            sns.heatmap(values,annot=ann,fmt='',cmap='RdBu_r',vmin=-limit,vmax=limit,center=0,annot_kws={'fontsize':26},cbar_kws={'label':'Delta (poin persentase) · persona − N'},ax=ax)
            for t in ax.texts:t.set_fontweight('bold' if '*' in t.get_text() else 'normal')
            ax.set_title(subset.title(),fontsize=26,fontweight='bold');ax.set_xlabel('');ax.set_ylabel('')
            ax.tick_params(axis='x',rotation=42,labelsize=22);ax.tick_params(axis='y',rotation=0,labelsize=22)
            ax.collections[0].colorbar.ax.tick_params(labelsize=22);ax.collections[0].colorbar.ax.yaxis.label.set_size(25)
        fig.suptitle('ETHICS · Persona agama: '+title,fontsize=30,fontweight='bold')
        fig.supxlabel('Persona − N · * p Holm <0,05 · keluarga 16 kontras/model/subset/outcome · n dan pasangan discordant di tabel',fontsize=16)
        stem=f'{number:02d}_ethics_religion_'+{'coverage':'coverage','conditional_accuracy':'conditional_accuracy','effective_success':'effective_success','stance_wrong':'stance_wrong'}[outcome]+'_effect_holm_heatmap'
        save_plot(fig,stem,'Parity vs N; p/efek asli')

render_parity_heatmaps()
manifest=publish()
print('NOTEBOOK_OK',NOTEBOOK_NAME)


PNG: 06_ethics_religion_accuracy_committed_valid_subsets_heatmap.png


PNG: 07_ethics_religion_effective_success_valid_subsets_heatmap.png


PNG: 08_ethics_religion_stance_wrong_construct_invalid_heatmap.png


PNG: 10_ethics_religion_coverage_effect_holm_heatmap.png


PNG: 11_ethics_religion_conditional_accuracy_effect_holm_heatmap.png


PNG: 12_ethics_religion_effective_success_effect_holm_heatmap.png


PNG: 13_ethics_religion_stance_wrong_effect_holm_heatmap.png


PUBLISH_OK ETHICS 33 PNG; sumber statistik tidak berubah.


NOTEBOOK_OK ETHICS_Religion_Accuracy_Heatmap_06_13.ipynb
